In [0]:
import subprocess
import os
import sys
import tempfile
import traceback
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()

def initialize_secrets():
    try:
        SECRET_SCOPE_NAME = os.environ.get("SECRET_SCOPE_NAME", "redox_oauth_keys")
        if not SECRET_SCOPE_NAME:
            raise ValueError("SECRET_SCOPE_NAME environment variable is required but was not set")
        print(f"[redox-proxy] Using secret scope: {SECRET_SCOPE_NAME}")
        print(f"[redox-proxy] Retrieving private key...")
        PRIVATE_KEY = w.secrets.get_secret(scope=SECRET_SCOPE_NAME, key="private_key").value
        print(f"[redox-proxy] Retrieving KID...")
        KID = w.secrets.get_secret(scope=SECRET_SCOPE_NAME, key="kid").value
        print(f"[redox-proxy] Retrieving client ID...")
        CLIENT_ID = w.secrets.get_secret(scope=SECRET_SCOPE_NAME, key="client_id").value
        os.environ["OAUTH_CLIENT_ID"] = CLIENT_ID
        os.environ["OAUTH_KEY_ID"] = KID
        print(f"[redox-proxy] Secrets retrieved and environment variables set")
        print(f"[redox-proxy] Writing private key to temp file...")
        temp_key_file = tempfile.NamedTemporaryFile(
            mode='w'
            , delete=False
            , suffix='.pem'
        )
        temp_key_file.write(PRIVATE_KEY)
        temp_key_file.close()
        os.chmod(temp_key_file.name, 0o600)
        os.environ["OAUTH_KEY_PATH"] = temp_key_file.name
        print(f"[redox-proxy] Private key written to: {temp_key_file.name}")
    except Exception as e:
        print(f"[redox-proxy] ERROR initializing secrets: {e}")
        print(f"[redox-proxy] Traceback: {traceback.format_exc()}")
        raise

initialize_secrets()

# Start the binary
proc = subprocess.Popen(
    ["/Volumes/mkgs/redox/bin/redox-mcp"],
    stdin=subprocess.PIPE,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True,
    env=os.environ.copy()
)

# Send initialize request
import json
request = {"jsonrpc": "2.0", "method": "initialize", "id": 1, "params": {"protocolVersion": "2025-11-25", "capabilities": {}}}
proc.stdin.write(json.dumps(request) + "\n")
proc.stdin.flush()

# Wait and check what comes back
import time
time.sleep(2)
print("Return code:", proc.poll())
print("Has stdout?", proc.stdout.readable())

# Try to read
import select
if select.select([proc.stdout], [], [], 1)[0]:
    print("STDOUT:", proc.stdout.readline())
else:
    print("No stdout available")

In [0]:
import json
import select
import time

# Send tools/list request to MCP server with client_info
tools_request = {
    "jsonrpc": "2.0"
    , "method": "tools/list"
    , "id": 2
    , "params": {
        "client_info": {
            "one": True
        }
    }
}

print("[redox-proxy] Sending tools/list request with client_info...")
proc.stdin.write(json.dumps(tools_request) + "\n")
proc.stdin.flush()

# Give it a moment to respond
time.sleep(1)

# Read the response
responses = []
while select.select([proc.stdout], [], [], 0.5)[0]:
    line = proc.stdout.readline().strip()
    if line:
        print(f"[redox-proxy] Raw response: {line}")
        try:
            response_data = json.loads(line)
            responses.append(response_data)
        except json.JSONDecodeError as e:
            print(f"[redox-proxy] Failed to parse JSON: {e}")
            print(f"[redox-proxy] Line content: {line}")

if responses:
    tools_response = responses[-1]  # Get the last response (should be tools/list)
    print(f"\n[redox-proxy] Tools response received")
    print(json.dumps(tools_response, indent=2))
else:
    print("[redox-proxy] No response received from MCP server")
    tools_response = None

In [0]:
import json
import select
import time

# Send connections/list request to MCP server for staging environment with client_info for databricks notebook
connections_request = {
    "jsonrpc": "2.0"
    , "method": "connections/list"
    , "id": 10
    , "params": {
        "environment": "staging"
        , "client_info": {
            "notebook": True
        }
    }
}

print("[redox-proxy] Sending connections/list request for staging environment with client_info for databricks notebook...")
proc.stdin.write(json.dumps(connections_request) + "\n")
proc.stdin.flush()

# Give it a moment to respond
time.sleep(1)

# Read the response
conn_responses = []
while select.select([proc.stdout], [], [], 0.5)[0]:
    line = proc.stdout.readline().strip()
    if line:
        print(f"[redox-proxy] Raw response: {line}")
        try:
            response_data = json.loads(line)
            conn_responses.append(response_data)
        except json.JSONDecodeError as e:
            print(f"[redox-proxy] Failed to parse JSON: {e}")
            print(f"[redox-proxy] Line content: {line}")

if conn_responses:
    connections_response = conn_responses[-1]
    print(f"\n[redox-proxy] Connections response received")
    print(json.dumps(connections_response, indent=2))
else:
    print("[redox-proxy] No response received from MCP server for connections in staging environment")
    connections_response = None

In [0]:
import json
import select
import time

# Send tools/list request to MCP server
tools_request = {
    "jsonrpc": "2.0"
    , "method": "tools/list"
    , "id": 2
    , "params": {}
}

print("[redox-proxy] Sending tools/list request...")
proc.stdin.write(json.dumps(tools_request) + "\n")
proc.stdin.flush()

# Give it a moment to respond
time.sleep(1)

# Read the response
responses = []
while select.select([proc.stdout], [], [], 0.5)[0]:
    line = proc.stdout.readline().strip()
    if line:
        print(f"[redox-proxy] Raw response: {line}")
        try:
            response_data = json.loads(line)
            responses.append(response_data)
        except json.JSONDecodeError as e:
            print(f"[redox-proxy] Failed to parse JSON: {e}")
            print(f"[redox-proxy] Line content: {line}")

if responses:
    tools_response = responses[-1]  # Get the last response (should be tools/list)
    print(f"\n[redox-proxy] Tools response received")
    print(json.dumps(tools_response, indent=2))
else:
    print("[redox-proxy] No response received from MCP server")
    tools_response = None

In [0]:
import json
import select
import time

# Send environments/list request to MCP server
environments_request = {
    "jsonrpc": "2.0"
    , "method": "environments/list"
    , "id": 3
    , "params": {}
}

print("[redox-proxy] Sending environments/list request...")
proc.stdin.write(json.dumps(environments_request) + "\n")
proc.stdin.flush()

# Give it a moment to respond
time.sleep(1)

# Read the response
env_responses = []
while select.select([proc.stdout], [], [], 0.5)[0]:
    line = proc.stdout.readline().strip()
    if line:
        print(f"[redox-proxy] Raw response: {line}")
        try:
            response_data = json.loads(line)
            env_responses.append(response_data)
        except json.JSONDecodeError as e:
            print(f"[redox-proxy] Failed to parse JSON: {e}")
            print(f"[redox-proxy] Line content: {line}")

if env_responses:
    environments_response = env_responses[-1]
    print(f"\n[redox-proxy] Environments response received")
    print(json.dumps(environments_response, indent=2))
else:
    print("[redox-proxy] No response received from MCP server for environments")
    environments_response = None

In [0]:
import json
import select
import time

# Send an authenticated request to MCP server (e.g., tools/list)
auth_test_request = {
    "jsonrpc": "2.0"
    , "method": "tools/list"
    , "id": 100
    , "params": {}
}

print("[redox-proxy] Sending authentication test request...")
proc.stdin.write(json.dumps(auth_test_request) + "\n")
proc.stdin.flush()

# Wait for response
time.sleep(1)

# Read the response
auth_responses = []
while select.select([proc.stdout], [], [], 0.5)[0]:
    line = proc.stdout.readline().strip()
    if line:
        print(f"[redox-proxy] Raw response: {line}")
        try:
            response_data = json.loads(line)
            auth_responses.append(response_data)
        except json.JSONDecodeError as e:
            print(f"[redox-proxy] Failed to parse JSON: {e}")
            print(f"[redox-proxy] Line content: {line}")

# Check authentication status
if auth_responses:
    last_response = auth_responses[-1]
    if 'result' in last_response:
        print("[redox-proxy] MCP server authentication confirmed: tools/list succeeded")
    elif 'error' in last_response:
        print(f"[redox-proxy] MCP server authentication failed: {last_response['error']}")
    else:
        print("[redox-proxy] Unexpected response format")
else:
    print("[redox-proxy] No response received from MCP server for authentication test")

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()

if tools_response and 'result' in tools_response:
    tools_data = json.dumps(tools_response['result'], indent=2)
    
    # Use a foundation model to analyze the tools
    prompt = f"""Analyze the following MCP server tools and provide a clear summary:

Tools available:
{tools_data}

Please provide:
1. A brief description of what this MCP server does based on the available tools
2. A list of each tool with its purpose
3. Key capabilities and use cases
"""
    
    print("[redox-proxy] Querying foundation model for tool analysis...\n")
    
    response = w.serving_endpoints.query(
        name="databricks-meta-llama-3-1-70b-instruct"
        , messages=[
            ChatMessage(
                role=ChatMessageRole.USER
                , content=prompt
            )
        ]
        , max_tokens=1000
    )
    
    analysis = response.choices[0].message.content
    print("=" * 80)
    print("FOUNDATION MODEL ANALYSIS")
    print("=" * 80)
    print(analysis)
    print("=" * 80)
else:
    print("[redox-proxy] No tools data available to analyze")